# Obtaining the cutouts of the plates

In [1]:
import json
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np
import random
import base64
import gzip
import requests
import os

##

In [2]:
index = {}

with open("../Step_2_Integration/observability_bright.json", "rb") as f:
    while True:
        pos = f.tell()
        line = f.readline()

        if not line:
            break

        s = line.strip()

        if s in (b"", b"{", b"}"):
            continue

        # key is before first colon
        key_bytes = s.split(b":", 1)[0]
        key = json.loads(key_bytes.decode())

        index[int(key)] = pos

print("nkeys:", len(index))

def load_observability_key(filename, index, key):
    key = int(key)

    with open(filename, "rb") as f:
        f.seek(index[key])
        line = f.readline().decode().strip()

    if line.endswith(","):
        line = line[:-1]

    rec = json.loads("{" + line + "}")
    data = np.asarray(rec[str(key)])
    return data

nkeys: 54355


In [3]:
mpcnum = 1 #start with Ceres
data = load_observability_key("../Step_2_Integration/observability_bright.json", index, mpcnum)
print(data)

[['243.33128045114415' '-12.93348140171074' '2.260736488440913' ...
  '2411433.917859954' 'i00768:0' '7.246961477225209']
 ['230.25215407460462' '-15.673388431403394' '2.193428991479598' ...
  '2411565.614578704' 'i01460:0' '7.265208046946477']
 ['339.79264463016915' '-19.64969621754664' '2.453585749902211' ...
  '2411898.912928241' 'b06327:0' '7.646977313385175']
 ...
 ['359.9835417431411' '-10.191137008676074' '3.0254352751584856' ...
  '2447528.49375' 'dnb06498:0' '8.059926518106346']
 ['95.49408964050082' '22.1566757774533' '2.1156381843676977' ...
  '2447823.74375' 'dnb06687:0' '7.1155268157385745']
 ['92.92281679962531' '24.41586281399997' '1.7395307135541211' ...
  '2447861.7277777777' 'dnb06705:0' '6.66692389543614']]


In [4]:
ra,dec,rearth,rsun,dradt,ddecdt,jd,platid,vmag = data.T

In [5]:
ra = data[:,0].astype(float)
dec = data[:,1].astype(float)
rearth = data[:,2].astype(float)
rsun = data[:,3].astype(float)
dradt = data[:,4].astype(float)
ddecdt = data[:,5].astype(float)
jd = data[:,6].astype(float)
plate_id, sol_id = np.array([entry.split(':') for entry in data[:,7]]).T
vmag = data[:,8].astype(float)

In [12]:
def retrieve_and_save(plate_id, sol_id, ra, dec, path="./"):
    os.makedirs(path, exist_ok=True)

    url = "https://api.starglass.cfa.harvard.edu/public/dasch/dr7/cutout"
    payload = {
        "plate_id": plate_id,
        "solution_number": int(sol_id),
        "center_ra_deg": float(ra),
        "center_dec_deg": float(dec),
    }

    r = requests.post(url, json=payload, headers={"Accept": "application/json"})
    r.raise_for_status()

    fits_bytes = gzip.decompress(base64.b64decode(r.json()))

    filename = f"{mpcnum}_{plate_id}.fits"
    filepath = os.path.join(path, filename)

    with open(filepath, "wb") as f:
        f.write(fits_bytes)

    return filepath

In [13]:
for i in range(100):
    filename = f"{mpcnum}_{plate_id[i]}.fits"
    filepath = os.path.join(str(mpcnum),filename,)

    if os.path.exists(filepath):
        print(f"Skipping {i}: {filename}")
        continue

    print(i)
    retrieve_and_save(plate_id[i],sol_id[i],ra[i],dec[i],path=str(mpcnum),)

Skipping 0: 1_i00768.fits
Skipping 1: 1_i01460.fits
Skipping 2: 1_b06327.fits
Skipping 3: 1_b06363.fits
Skipping 4: 1_b06387.fits
Skipping 5: 1_b06388.fits
Skipping 6: 1_b06561.fits
Skipping 7: 1_i07633.fits
Skipping 8: 1_i07647.fits
Skipping 9: 1_i07678.fits
Skipping 10: 1_i07723.fits
Skipping 11: 1_i07825.fits
Skipping 12: 1_i07880.fits
Skipping 13: 1_a00379.fits
Skipping 14: 1_i11021.fits
Skipping 15: 1_b13539.fits
Skipping 16: 1_b13580.fits
Skipping 17: 1_b13728.fits
Skipping 18: 1_b13936.fits
Skipping 19: 1_b14045.fits
Skipping 20: 1_b14745.fits
Skipping 21: 1_b15043.fits
Skipping 22: 1_b15954.fits
Skipping 23: 1_b15996.fits
Skipping 24: 1_b16333.fits
Skipping 25: 1_b16462.fits
Skipping 26: 1_b16839.fits
Skipping 27: 1_b16924.fits
Skipping 28: 1_i15663.fits
Skipping 29: 1_i15821.fits
Skipping 30: 1_i15911.fits
Skipping 31: 1_b17469.fits
Skipping 32: 1_b17575.fits
Skipping 33: 1_b17594.fits
Skipping 34: 1_b17637.fits
Skipping 35: 1_i16201.fits
Skipping 36: 1_b17732.fits
37
38
39
40